# Title 1: Frequency and timing of dietitian assessment in critically ill patients with kidney disease
**MIMIC-III v1.4, MetaVision stays (2008-2012)**

**Data rules (PhysioNet DUA)**
- Run only on your own computer. This notebook reads the **times** of Nutrition notes; the `TEXT` column is never loaded.
- Only files in `outputs/` whose names start with `summary_t1_` may be shared (aggregate counts; 1-10 shown as `<11`).

**All files go in `C:\Eksperimen\mimic\gizi`** (results in its `outputs` subfolder): ICUSTAYS, PATIENTS, ADMISSIONS, DIAGNOSES_ICD, PROCEDURES_ICD, PROCEDUREEVENTS_MV, D_ITEMS, NOTEEVENTS, INPUTEVENTS_MV (`.csv.gz`), plus `mimic_t3.py`, `mimic_t1.py`, `formula_composition.csv`.

In [ ]:
import os, sys
from pathlib import Path
# All MIMIC files, the .py modules and the .csv tables are in C:\Eksperimen\mimic\gizi
# (seen from WSL as /mnt/c/Eksperimen/mimic/gizi).
CANDIDATES = [os.environ.get('MIMIC_DIR', ''), '/mnt/c/Eksperimen/mimic/gizi', r'C:\Eksperimen\mimic\gizi']
DATA = next((Path(p) for p in CANDIDATES if p and Path(p).is_dir()), None)
assert DATA is not None, r'Folder C:\Eksperimen\mimic\gizi not found; check the path'
sys.path.insert(0, str(DATA))            # so that mimic_t3.py / mimic_t1.py are found
OUT = DATA / 'outputs'; OUT.mkdir(exist_ok=True)
print('Data folder:', DATA)
print('Output folder:', OUT)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import mimic_t3 as m3
import mimic_t1 as m1
need = ['ICUSTAYS','PATIENTS','ADMISSIONS','DIAGNOSES_ICD','PROCEDURES_ICD','PROCEDUREEVENTS_MV',
        'D_ITEMS','NOTEEVENTS','INPUTEVENTS_MV']
print('Missing files:', [f for f in need if not (DATA/f'{f}.csv.gz').exists()] or 'none')

## Step 1. Cohort (ICU stay of 24 h or more) and kidney groups

In [ ]:
m3.MIN_LOS_DAYS = 1                       # keep stays of 24 h or more; primary cohort is 48 h or more
icu = m3.load_icustays(DATA)
cohort, flow = m3.base_cohort(icu)
dx = m3.kidney_dx(DATA, cohort.HADM_ID)
pf7, labels = m3.procedure_flags(DATA, cohort)              # KRT groups: up to end of ICU day 7
pf48, _ = m3.procedure_flags(DATA, cohort, hours=48)        # ventilation in the first 48 h
cohort = (cohort.merge(dx, on='HADM_ID', how='left')
                .merge(pf7, on='ICUSTAY_ID', how='left')
                .merge(pf48[['ICUSTAY_ID','VENT']].rename(columns={'VENT':'VENT48'}), on='ICUSTAY_ID', how='left'))
for c in ['AKI','CKD','ESRD','KRT_ICD','CKRT_EVT','IKRT_EVT','VENT','VENT48']:
    cohort[c] = cohort[c].fillna(False).astype(bool)
cohort = m3.assign_group(cohort)
cohort.GROUP.value_counts()

## Step 2. Dietitian assessment from Nutrition note times

In [ ]:
# Reads NOTEEVENTS in chunks without the TEXT column; takes several minutes.
notes = m1.nutrition_note_times(DATA, cohort.HADM_ID)
cohort = cohort.merge(m1.assessment_by_stay(cohort, notes), on='ICUSTAY_ID', how='left')
cohort = m1.add_times(cohort)
src = cohort.FIRST_SOURCE.value_counts().rename_axis('time_source').reset_index(name='first_notes')
src['first_notes'] = src.first_notes.map(m3.suppress)
m3.save_summary(src, OUT, 'summary_t1_timesource')
src

In [ ]:
primary = cohort[cohort.LOS_H >= 48].reset_index(drop=True)
f = flow + [('ICU stay 48 h or longer (primary cohort)', len(primary), primary.SUBJECT_ID.nunique())]
flow_df = pd.DataFrame(f, columns=['step','icu_stays','patients'])
flow_df[['icu_stays','patients']] = flow_df[['icu_stays','patients']].map(m3.suppress)
m3.save_summary(flow_df, OUT, 'summary_t1_flow')
t1 = m1.table1(primary); m3.save_summary(t1, OUT, 'summary_t1_table1')
flow_df

## Step 3. Frequency and timing of assessment

In [ ]:
asum = m1.assessment_summary(primary)
m3.save_summary(asum, OUT, 'summary_t1_assessment')
cif = m1.cumulative_incidence(primary)
m3.save_summary(cif, OUT, 'summary_t1_cif')
asum.T

## Step 4. Factors associated with assessment within 48 h

In [ ]:
reg = m1.regression(primary, 'WITHIN_48H')
m3.save_summary(reg, OUT, 'summary_t1_logit')
reg

## Step 5. Assessment relative to the start of enteral nutrition

In [ ]:
comp = pd.read_csv(DATA / 'formula_composition.csv')
items = m3.nutrition_items(DATA, comp)
rows, _ = m3.scan_inputevents(DATA, primary, items)      # one pass over INPUTEVENTS_MV
r = m3.nutrient_amounts(rows, items)
primary = primary.drop(columns=[c for c in ['FIRST_EN'] if c in primary]).merge(
    m1.first_en_start(r), on='ICUSTAY_ID', how='left')
link = m1.en_link(primary)
m3.save_summary(link, OUT, 'summary_t1_en_link')
link.T

## Step 6. Sensitivity analyses

In [ ]:
def share(c, flag, label):
    out = []
    for grp, s in m1.by_group(c):
        n = len(s)
        out.append({'analysis': label, 'group': grp, 'n': m3.suppress(n),
                    'assessed_pct_ci': m1.pct_ci(int(flag.loc[s.index].sum()), n) if n >= m3.SMALL else ''})
    return pd.DataFrame(out)

c24 = cohort[cohort.LOS_H >= 24]; c72 = cohort[cohort.LOS_H >= 72]
no_dateonly = primary[primary.FIRST_SOURCE.fillna('') != 'CHARTDATE only']
sens = pd.concat([
    share(primary, primary.WITHIN_48H, 'Primary: stay 48 h or more, assessed within 48 h'),
    share(c24, c24.WITHIN_24H, 'Stay 24 h or more, assessed within 24 h'),
    share(c72, c72.WITHIN_72H, 'Stay 72 h or more, assessed within 72 h'),
    share(primary, primary.WITHIN_48H | primary.PRE_ICU_NOTE, 'Counting assessment before ICU admission'),
    share(no_dateonly, no_dateonly.WITHIN_48H, 'Excluding first notes timed by chart date only'),
], ignore_index=True)
m3.save_summary(sens, OUT, 'summary_t1_sensitivity')
sens

## Step 7. Figure (aggregate only)

In [ ]:
COL = {'No kidney disease':'#0072B2','Kidney disease without KRT':'#E69F00','Intermittent KRT':'#009E73','CKRT':'#CC79A7'}
fig, ax = plt.subplots(figsize=(7, 4.2))
for g, s in cif[cif.group != 'All'].groupby('group'):
    ax.step(s.hour, s.cum_incidence_pct, where='post', lw=2, color=COL[g], label=g)
    ax.fill_between(s.hour, s.ci_low, s.ci_high, step='post', color=COL[g], alpha=.12, lw=0)
for h in (24, 48):
    ax.axvline(h, color='0.75', lw=1, ls='--', zorder=0)
ax.set(xlabel='Hours since ICU admission', ylabel='Cumulative incidence of first\ndietitian assessment (%)',
       xlim=(0, 168), ylim=(0, 100), xticks=range(0, 169, 24))
ax.spines[['top','right']].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc='lower right')
fig.tight_layout(); fig.savefig(OUT/'summary_t1_fig1_cumulative_incidence.png', dpi=300)

## What to send
Only the files in `outputs/` that start with `summary_t1_`: flow, table1, timesource, assessment, cif, logit, en_link, sensitivity (.csv) and `summary_t1_fig1_cumulative_incidence.png`.

In [ ]:
sorted(p.name for p in OUT.iterdir() if p.name.startswith('summary_t1_'))